In [1]:
using Random, Distributions


## Model Parameter and function setup


In [2]:
N = 3 # Countries 
J = 20000 # Number of Goods 
sigma = 2 # Elasticity 
theta = 4 # Dispersion Parameter (Comparative Advantage)
T = ones(N, 1) * 1.5 # Technology (Absolute Advantage)
tau = ones(N, N) # Trade Cost 
L = ones(N, 1) # Labor Force 
w = ones(N, 1) # Wages 

Random.seed!(1234)

TaskLocalRNG()

In [3]:
function draw_prods(N, J, theta, T) # Draws our productivities using the CDF from the EK paper 
    U = rand(J, N)
    return (-T' ./ log.(U)).^(1/theta) 
end 

z = draw_prods(N, J, theta, T) 

20000×3 Matrix{Float64}:
 1.28804   0.85982   1.01292
 1.1399    1.62592   1.01127
 2.69918   1.26942   0.91342
 0.772789  1.53782   1.34379
 1.23099   2.01918   1.75115
 1.35348   1.30513   1.3829
 1.71153   1.34728   1.05382
 2.58855   1.12312   1.0841
 1.58776   1.13251   1.61455
 1.4264    1.78731   1.28068
 ⋮                   
 1.09128   1.10697   1.74032
 1.67914   1.30324   1.16063
 0.998359  1.74091   1.06587
 2.20369   1.54409   1.11671
 1.33779   1.08176   1.61259
 0.95708   0.795476  1.19971
 1.35274   0.932732  0.896956
 1.01939   1.37687   2.38477
 2.0511    1.65483   1.17922

In [4]:
function find_trade_shares(z, tau, w) 
    costs = zeros(J, N, N)
    for n in 1:N
        for i in 1:N
            costs[:, n, i] = tau[n, i] * w[i] ./ z[:, i]
        end 
    end 
    trade_shares = zeros(N,N)
    for n in 1:N 
        for i in 1:N
            trade_shares[n, i] = mean([argmin(costs[k, n, :]) == i for k in 1:J])
        end
    end
    return trade_shares 
end

trade_shares = find_trade_shares(z, tau, w) 


3×3 Matrix{Float64}:
 0.33255  0.33025  0.3372
 0.33255  0.33025  0.3372
 0.33255  0.33025  0.3372

In [5]:
function trade_bals(trade_shares, L, w)
    expenditure = L .* w 
    trade_matrix = trade_shares .* expenditure' # Broadcast the expenditure across the rows of trade shares 
    trade_balances = zeros(N) 
    for n in 1:N 
         exports = sum(trade_matrix[:, n])
         imports = sum(trade_matrix[n, :])
         trade_balances[n] = exports - imports
    end 
    return trade_balances
end

trade_balances = trade_bals(trade_shares, L, w)

3-element Vector{Float64}:
 -0.0023499999999999632
 -0.00924999999999998
  0.011600000000000055

In [6]:
function equilibrium_info(sigma, z, tau, L, tol = 1e-4, max_iter = 10000)
    w = ones(N)

    for iter in 1:max_iter
        trade_shares = find_trade_shares(z, tau, w)
        trade_balances = trade_bals(trade_shares, L, w)
        if abs(maximum(trade_balances)) < tol
            break 
        end
        w[2:end] .+= .01 .* trade_balances[2:end] ./ (L[2:end] .* w[2:end]) #adjust wages 2 and 3 based on imbalances in trade balances
    end
    println("Equilibrium Wages: ", w)

    costs = zeros(J, N, N)
    for n in 1:N
        for i in 1:N
            costs[:, n, i] = tau[n, i] * w[i] ./ z[:, i]
        end 
    end 
    p = [mean((minimum(costs[:, n, :], dims = 2).^(1 - sigma))).^(1 / (1-sigma)) for n in 1:N]

    println("Price Indices: ", p)

    welfare = (w .* L) ./ p

    println("Welfare: ", welfare)

    return w, welfare

end

equilibrium_info (generic function with 3 methods)

In [7]:
equilibrium_info(sigma, z, tau, L)

Equilibrium Wages: [1.0, 0.9974498608454807, 1.0034458447962582]
Price Indices: [0.562568775124069, 0.562568775124069, 0.562568775124069]
Welfare: [1.7775604409957872; 1.7730274145156795; 1.783685638391427;;]


([1.0, 0.9974498608454807, 1.0034458447962582], [1.7775604409957872; 1.7730274145156795; 1.783685638391427;;])

# The output from the above cell is the solution to part A

# Part B
- Symmetric Geography, $\tau = 1.1$ for all $i \neq j$

In [8]:
tau = [n == i ? 1.0 : 1.1 for n in 1:N, i in 1:N]
trade_shares_b = find_trade_shares(z, tau, w)

3×3 Matrix{Float64}:
 0.4245   0.28385  0.29165
 0.2896   0.418    0.2924
 0.28745  0.2854   0.42715

Every country now supplies a significantly larger share of their own goods as foreign productivity draws may not be able to overcome the trade cost. 

In [9]:
equilibrium_info(sigma, z, tau, L)

Equilibrium Wages: [1.0, 0.995226337655505, 1.0029971462920357]
Price Indices: [0.5965396186784323, 0.5961958686360354, 0.5964863735770873]
Welfare: [1.6763345948679647; 1.6692942538034745; 1.6815089006595936;;]


([1.0, 0.995226337655505, 1.0029971462920357], [1.6763345948679647; 1.6692942538034745; 1.6815089006595936;;])

All that seems to change in the equilibrium is that price indices are higher and welfare is lower across the board. Wages remain unchanged.

# Part C
- Asymmetric Geography, country 3 is further away from countries 1 and 2, leading to a higher trade cost. 

In [10]:
tau = ones(N, N)
tau[:, 3] .= 1.3 
tau[3, :] .= 1.3
tau[3, 3] = 1
tau[1, 2] = 1.05
tau[2, 1] = 1.05 
tau

3×3 Matrix{Float64}:
 1.0   1.05  1.3
 1.05  1.0   1.3
 1.3   1.3   1.0

In [11]:
welfare_c, w_c = equilibrium_info(sigma, z, tau, L)

Equilibrium Wages: [1.0, 0.9976554821877067, 0.9521548985201885]
Price Indices: [0.6041435854599984, 0.6040167753663552, 0.6280122912015462]
Welfare: [1.655235649383903; 1.6517016130596658; 1.5161405467056632;;]


([1.0, 0.9976554821877067, 0.9521548985201885], [1.655235649383903; 1.6517016130596658; 1.5161405467056632;;])

It can be seen that the higher costs of trade for country 3 lead to lower real wages and welfare. 

In [12]:
trade_shares_c = find_trade_shares(z, tau, w_c) 

3×3 Matrix{Float64}:
 0.42995  0.35395  0.2161
 0.35465  0.4287   0.21665
 0.1652   0.1656   0.6692

It can also be seen that Country 3 produces a significantly larger proportion of their own goods compared to countries 1 and 2. 

# Part D
- Country 2 has an improvement in technology, now being $T_2 = 3$
- Everything else is the same as Part C

In [13]:
T_d = [1.0, 3.0, 1.0]

z_d = draw_prods(N, J, theta, T_d) 

20000×3 Matrix{Float64}:
 1.0983    1.40725   0.880356
 1.11029   0.995769  1.48869
 1.36499   1.88486   1.14786
 1.13944   1.07938   1.01665
 0.796578  1.30329   1.33246
 1.1168    4.39224   1.2883
 1.54924   1.03191   1.26107
 1.11527   1.51586   1.05872
 2.24975   1.27421   0.735206
 1.08256   1.05262   1.12379
 ⋮                   
 0.845334  1.06941   1.23819
 1.37764   0.84247   0.868702
 1.42956   1.38356   1.14595
 1.55889   1.46663   0.698257
 1.26721   1.48015   0.862512
 1.31038   1.51098   0.938633
 1.814     1.18417   1.10557
 2.39528   1.18319   0.914152
 1.10074   1.48578   1.05503

In [14]:
welfare_d, w_d = equilibrium_info(sigma, z_d, tau, L)

Equilibrium Wages: [1.0, 1.4219096780591578, 0.966472537649179]
Price Indices: [0.6872340682397758, 0.6915160394226512, 0.7084209136841645]
Welfare: [1.4551083047458877; 2.0562208206281296; 1.364263136477732;;]


([1.0, 1.4219096780591578, 0.966472537649179], [1.4551083047458877; 2.0562208206281296; 1.364263136477732;;])

The increase in productivity in country 2 causes wages and welfare to increase in country 2 as would be expected. The main surprise here is that countries 1 and 3 seem to be worse off in the welfare department, which is counterintuitive. 

In [15]:
find_trade_shares(z_d, tau, w_d) 

3×3 Matrix{Float64}:
 0.4844   0.29365  0.22195
 0.4057   0.36595  0.22835
 0.18125  0.1352   0.68355

Countries 1 and 2 seem to produce more of their own goods now that country 2 is more productive, which seems counterintuitive. Another thing worth npoting is country 2 produces less of its own goods than it did in the case before its productivity was increased. Most of this was transferred to buying from country 1, as there is a lower trade cost between the two countries. 

# Part E
- Changing the Frechet dispersion. 
- $\theta = 8$
- Other than this, same setup as part c.

In [16]:
theta = 8

z_e = draw_prods(N, J, theta, T) 


20000×3 Matrix{Float64}:
 0.973494  1.47021   1.10514
 1.04158   1.19383   1.31824
 0.898584  0.857986  0.971503
 0.999768  1.22777   1.02097
 0.993789  0.980875  1.09697
 1.0632    1.20327   1.22799
 1.07983   1.02732   1.22511
 1.01187   1.36469   1.15048
 0.943779  1.1607    1.16524
 0.98683   1.09687   1.00584
 ⋮                   
 1.14437   1.24215   1.06017
 0.962242  1.17037   1.19045
 1.1793    1.21358   0.960797
 0.952513  1.04575   1.21016
 1.02429   1.04354   1.46197
 1.13254   0.89509   0.866305
 0.826069  1.12687   1.19332
 0.980916  1.21885   1.57347
 1.12351   1.13722   1.23189

In [17]:
welfare_e, w_e = equilibrium_info(sigma, z_e, tau, L)

Equilibrium Wages: [1.0, 0.9996087059659342, 0.9729909016282606]
Price Indices: [0.8112192169018708, 0.8112746768183063, 0.8308621641846519]
Welfare: [1.2327124150474422; 1.2321458249951112; 1.1710617519610893;;]


([1.0, 0.9996087059659342, 0.9729909016282606], [1.2327124150474422; 1.2321458249951112; 1.1710617519610893;;])

In [18]:
trade_shares_e = find_trade_shares(z_e, tau, w_e) 

3×3 Matrix{Float64}:
 0.54135  0.36075  0.0979
 0.36115  0.5392   0.09965
 0.06585  0.0685   0.86565

Welfare across the board is significantly lower compared to the economy in part c. The main intuition behind this result is that larger draws are now more rare, meaning there are far less goods where one country has a significantly better draw than the other two. This means that each country is now producing much more of it's own goods as seen in the trade share matrix above. This is because there are less draws where comparative advantage is able to overcome the costs associated with trade between countries.